# Notebook 2: Consolidación, Modelos Econométricos y Clustering
## Reto 2: Seguridad y Obras Públicas — Bogotá
### Concurso de Datos por la Seguridad — CCB

---

**Objetivo:** Consolidar las bases limpias, estimar modelos econométricos para probar hipótesis,
y aplicar clustering para identificar perfiles de tramos del Metro por seguridad.

**Estructura:**
1. Carga de datos limpios (del Notebook 01)
2. Estadísticas descriptivas consolidadas
3. Hipótesis y modelos econométricos
4. Clustering de tramos del Metro
5. Recomendaciones basadas en datos

In [3]:
# =============================================================================
# 0. CONFIGURACIÓN
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

# Directorio de datos limpios (salida del Notebook 01)
DATA_DIR = './datos_limpios'
print("✅ Librerías cargadas")

✅ Librerías cargadas


In [ ]:
# =============================================================================
# 1. CARGA DE DATOS LIMPIOS
# =============================================================================
corredores = pd.read_csv(f'{DATA_DIR}/corredores_limpio.csv')
ecn = pd.read_csv(f'{DATA_DIR}/ecn_2024_bogota.csv')
emu = pd.read_csv(f'{DATA_DIR}/emu_2024_filtrado.csv')
epv = pd.read_csv(f'{DATA_DIR}/epv_2025_filtrado.csv')
rnmc_loc = pd.read_csv(f'{DATA_DIR}/rnmc_indicadores_localidad.csv')
rnmc_upz = pd.read_csv(f'{DATA_DIR}/rnmc_indicadores_upz.csv')

print("📦 Datasets cargados:")
for nombre, df in [('Corredores', corredores), ('ECN', ecn), ('EMU', emu), 
                    ('EPV', epv), ('RNMC Localidad', rnmc_loc), ('RNMC UPZ', rnmc_upz)]:
    print(f"  {nombre}: {df.shape}")

---
## PARTE I: ESTADÍSTICAS DESCRIPTIVAS CONSOLIDADAS

Antes de modelar, entendamos los datos.

In [ ]:
# =============================================================================
# 2.1 Corredores: Comparación Metro PLM vs Calle 13
# =============================================================================

print("=" * 70)
print("COMPARACIÓN METRO PLM vs CALLE 13")
print("=" * 70)

# Variables a comparar
vars_comparar = {
    'P33G': 'Satisfacción con seguridad (1-5)',
    'seguridad_empeoro': '% Seguridad empeoró',
    'fue_victima': '% Fue víctima de delito',
    'indice_satisfaccion': 'Índice satisfacción entorno',
    'P37': 'Acuerdo con el plan (1-5)',
    'P40': 'Preocupación por duración (1-5)',
}

comparacion = []
for var, label in vars_comparar.items():
    if var in corredores.columns:
        metro = corredores.loc[corredores['TRAMO'] == 2, var]
        calle13 = corredores.loc[corredores['TRAMO'] == 1, var]
        
        # Test de diferencia de medias
        t_stat, p_val = stats.ttest_ind(metro.dropna(), calle13.dropna())
        
        comparacion.append({
            'Variable': label,
            'Metro PLM': metro.mean(),
            'Calle 13': calle13.mean(),
            'Diferencia': metro.mean() - calle13.mean(),
            't-stat': t_stat,
            'p-valor': p_val,
            'Significativa': '***' if p_val < 0.01 else ('**' if p_val < 0.05 else ('*' if p_val < 0.1 else ''))
        })

df_comp = pd.DataFrame(comparacion)
print(df_comp.to_string(index=False, float_format='%.3f'))

In [ ]:
# =============================================================================
# 2.2 Corredores: Análisis por segmento del Metro
# =============================================================================

# Solo establecimientos en el corredor del Metro
metro_df = corredores[corredores['TRAMO'] == 2].copy()

# Estadísticas por segmento
vars_por_segmento = ['P33G', 'seguridad_empeoro', 'fue_victima', 'indice_satisfaccion']
vars_disponibles = [v for v in vars_por_segmento if v in metro_df.columns]

segmento_stats = metro_df.groupby('PRIMERA')[vars_disponibles].mean()

fig, axes = plt.subplots(1, len(vars_disponibles), figsize=(5*len(vars_disponibles), 6))
if len(vars_disponibles) == 1:
    axes = [axes]

labels_vars = {
    'P33G': 'Satisf. Seguridad',
    'seguridad_empeoro': '% Seg. Empeoró',
    'fue_victima': '% Víctima',
    'indice_satisfaccion': 'Satisf. Entorno',
}

for i, var in enumerate(vars_disponibles):
    segmento_stats[var].sort_values().plot(kind='barh', ax=axes[i], color='steelblue')
    axes[i].set_title(labels_vars.get(var, var))
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Segmento (PRIMERA)')

plt.suptitle('Indicadores por Segmento del Metro PLM', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig_03_segmentos_metro.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Figura guardada: fig_03_segmentos_metro.png")

In [ ]:
# =============================================================================
# 2.3 Comparación MiPymes vs Grandes empresas
# =============================================================================

print("=" * 70)
print("COMPARACIÓN MiPymes vs GRANDES EMPRESAS")
print("=" * 70)

# En corredores
if 'es_mipyme' in corredores.columns:
    print("\n--- CORREDORES ESTRATÉGICOS ---")
    for var, label in vars_comparar.items():
        if var in corredores.columns:
            mipyme = corredores.loc[corredores['es_mipyme'] == 1, var]
            grande = corredores.loc[corredores['es_mipyme'] == 0, var]
            if len(grande.dropna()) > 5:  # Solo si hay suficientes grandes empresas
                t_stat, p_val = stats.ttest_ind(mipyme.dropna(), grande.dropna())
                print(f"{label}: MiPyme={mipyme.mean():.3f}, Grande={grande.mean():.3f}, p={p_val:.3f}")

# En ECN
if 'es_mipyme' in ecn.columns and 'P56_1' in ecn.columns:
    print("\n--- ECN 2024 (BOGOTÁ) ---")
    print(ecn.groupby('tamano_empresa')[['P56_1']].agg(['mean', 'count']).round(3))

In [ ]:
# =============================================================================
# 2.4 Evolución temporal RNMC
# =============================================================================

# Tendencia de comparendos de seguridad por año
tendencia = rnmc_loc.groupby('ANIO')[['total_comparendos', 'comparendos_seguridad', 'comparendos_movilidad']].sum()

fig, ax = plt.subplots(figsize=(10, 5))
tendencia.plot(ax=ax, marker='o')
ax.set_title('Evolución de Comparendos en Bogotá — RNMC 2020–2025')
ax.set_ylabel('Número de comparendos')
ax.set_xlabel('Año')
ax.legend(['Total', 'Seguridad', 'Movilidad'])
plt.tight_layout()
plt.savefig('fig_04_tendencia_rnmc.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Figura guardada: fig_04_tendencia_rnmc.png")

---
## PARTE II: HIPÓTESIS Y MODELOS ECONOMÉTRICOS

### Hipótesis a probar:

**H1:** Los establecimientos en el corredor del Metro PLM reportan peor percepción de seguridad que los de la Calle 13.

**H2:** Las MiPymes perciben mayor inseguridad y son más vulnerables que las grandes empresas en los corredores de obras.

**H3:** Los factores del entorno (iluminación, señalización, presencia policial) están asociados significativamente con la percepción de seguridad.

**H4:** La victimización está correlacionada con la percepción de deterioro de seguridad desde el inicio de obras.

### Modelos:
1. **Logit/Probit:** P(seguridad_empeoro = 1) = f(corredor, tamaño, sector, factores entorno)
2. **Logit:** P(fue_victima = 1) = f(corredor, tamaño, sector, satisfacción entorno)
3. **OLS:** P33G (satisfacción seguridad) = f(corredor, tamaño, factores, percepción obras)
4. **Ordered Logit:** P43 (seguridad mejoró/empeoró/igual) = f(factores del entorno)

In [ ]:
# =============================================================================
# 3.1 Modelo 1: Logit — ¿Qué determina que la seguridad empeore?
# =============================================================================
print("=" * 70)
print("MODELO 1: Logit — P(Seguridad empeoró) = f(X)")
print("=" * 70)

# Preparar datos
modelo1_vars = ['TRAMO', 'es_mipyme', 'F5', 'P33G', 'P40',
                'P43_1A', 'P43_1B', 'P43_1C', 'P43_1D', 'P43_1E', 'P43_1F', 'P43_1G']
modelo1_vars_exist = [v for v in modelo1_vars if v in corredores.columns]

df_modelo1 = corredores[['seguridad_empeoro'] + modelo1_vars_exist].dropna()

if len(df_modelo1) > 50:
    # Variable dependiente
    y = df_modelo1['seguridad_empeoro']
    
    # Crear dummy para corredor Metro
    df_modelo1['es_metro'] = (df_modelo1['TRAMO'] == 2).astype(int)
    
    # Variables independientes
    X_vars = ['es_metro', 'es_mipyme']
    # Agregar factores de entorno si existen
    factores_entorno = ['P43_1A', 'P43_1B', 'P43_1C', 'P43_1D', 'P43_1E', 'P43_1F', 'P43_1G']
    X_vars += [v for v in factores_entorno if v in df_modelo1.columns]
    
    X = sm.add_constant(df_modelo1[X_vars])
    
    # Estimar Logit
    try:
        logit_model = Logit(y, X)
        logit_result = logit_model.fit(disp=0)
        print(logit_result.summary())
        
        # Efectos marginales
        print("\n--- EFECTOS MARGINALES (en la media) ---")
        margeff = logit_result.get_margeff()
        print(margeff.summary())
    except Exception as e:
        print(f"⚠️ Error en el modelo: {e}")
        print("Intente con un modelo OLS como alternativa...")
        ols_result = sm.OLS(y, X).fit()
        print(ols_result.summary())
else:
    print(f"⚠️ Datos insuficientes para el modelo: {len(df_modelo1)} observaciones")

In [ ]:
# =============================================================================
# 3.2 Modelo 2: Logit — ¿Qué determina la victimización?
# =============================================================================
print("=" * 70)
print("MODELO 2: Logit — P(Víctima) = f(X)")
print("=" * 70)

modelo2_vars = ['TRAMO', 'es_mipyme', 'F5', 'P33G', 'indice_satisfaccion', 'P40']
modelo2_vars_exist = [v for v in modelo2_vars if v in corredores.columns]

df_modelo2 = corredores[['fue_victima'] + modelo2_vars_exist].dropna()

if len(df_modelo2) > 50:
    y2 = df_modelo2['fue_victima']
    df_modelo2['es_metro'] = (df_modelo2['TRAMO'] == 2).astype(int)
    
    X2_vars = ['es_metro', 'es_mipyme']
    if 'P33G' in df_modelo2.columns:
        X2_vars.append('P33G')
    if 'indice_satisfaccion' in df_modelo2.columns:
        X2_vars.append('indice_satisfaccion')
    if 'P40' in df_modelo2.columns:
        X2_vars.append('P40')
    
    X2 = sm.add_constant(df_modelo2[X2_vars])
    
    try:
        logit2 = Logit(y2, X2).fit(disp=0)
        print(logit2.summary())
        print("\n--- EFECTOS MARGINALES ---")
        print(logit2.get_margeff().summary())
    except Exception as e:
        print(f"⚠️ Error: {e}")
        ols2 = sm.OLS(y2, X2).fit()
        print(ols2.summary())
else:
    print(f"⚠️ Datos insuficientes: {len(df_modelo2)} obs.")

In [ ]:
# =============================================================================
# 3.3 Modelo 3: OLS — Satisfacción con seguridad
# =============================================================================
print("=" * 70)
print("MODELO 3: OLS — Satisfacción seguridad (P33G) = f(X)")
print("=" * 70)

if 'P33G' in corredores.columns:
    modelo3_vars = ['TRAMO', 'es_mipyme', 'P37', 'P40', 'fue_victima',
                    'P43_1B', 'P43_1F', 'P43_1G']
    modelo3_vars_exist = [v for v in modelo3_vars if v in corredores.columns]
    
    df_modelo3 = corredores[['P33G'] + modelo3_vars_exist].dropna()
    
    if len(df_modelo3) > 50:
        y3 = df_modelo3['P33G']
        df_modelo3['es_metro'] = (df_modelo3['TRAMO'] == 2).astype(int)
        
        X3_cols = ['es_metro', 'es_mipyme'] + [v for v in modelo3_vars_exist if v not in ['TRAMO', 'es_mipyme']]
        X3 = sm.add_constant(df_modelo3[X3_cols])
        
        ols3 = sm.OLS(y3, X3).fit(cov_type='HC1')  # Errores robustos
        print(ols3.summary())
    else:
        print(f"⚠️ Datos insuficientes: {len(df_modelo3)} obs.")

In [ ]:
# =============================================================================
# 3.4 Modelo 4 (ECN): Seguridad empresarial en Bogotá
# =============================================================================
print("=" * 70)
print("MODELO 4: Logit — P(Víctima ECN) = f(tamaño, sector, infraestructura)")
print("=" * 70)

if 'P58' in ecn.columns:
    ecn_modelo = ecn[['P58', 'F5', 'F4', 'P33_C', 'P33_H', 'P33_S', 'P33_W', 'P32']].dropna()
    
    if len(ecn_modelo) > 50:
        # P58: víctima de delito (necesita ser binaria)
        y4 = (ecn_modelo['P58'] == 1).astype(int)  # Ajustar según codificación
        
        ecn_modelo['es_mipyme'] = ecn_modelo['F5'].isin([1, 2, 3]).astype(int)
        ecn_modelo['es_comercio'] = (ecn_modelo['F4'] == 2).astype(int)
        ecn_modelo['es_servicios'] = (ecn_modelo['F4'] == 3).astype(int)
        
        X4_cols = ['es_mipyme', 'es_comercio', 'es_servicios', 'P33_C', 'P33_H', 'P33_S', 'P32']
        X4_exist = [v for v in X4_cols if v in ecn_modelo.columns]
        X4 = sm.add_constant(ecn_modelo[X4_exist])
        
        try:
            logit4 = Logit(y4, X4).fit(disp=0)
            print(logit4.summary())
            print("\n--- EFECTOS MARGINALES ---")
            print(logit4.get_margeff().summary())
        except Exception as e:
            print(f"⚠️ Error: {e}")
    else:
        print(f"⚠️ Datos insuficientes: {len(ecn_modelo)} obs.")

---
## PARTE III: CLUSTERING DE TRAMOS DEL METRO

**Objetivo:** Identificar grupos de tramos del Metro con perfiles similares de seguridad
para proponer intervenciones diferenciadas.

**Variables para clustering:**
- Satisfacción con seguridad (P33G)
- % que reporta empeoramiento de seguridad
- % víctimas de delito
- Factores de entorno (iluminación, señalización, vigilancia, etc.)
- Índice de satisfacción general

In [ ]:
# =============================================================================
# 4.1 Preparar datos para clustering
# =============================================================================

# Agregar por segmento del Metro
metro_df = corredores[corredores['TRAMO'] == 2].copy()

# Variables para el perfil de cada segmento
vars_cluster_base = ['P33G', 'seguridad_empeoro', 'fue_victima', 'indice_satisfaccion',
                     'P43_1A', 'P43_1B', 'P43_1C', 'P43_1D', 'P43_1E', 'P43_1F', 'P43_1G']
vars_cluster = [v for v in vars_cluster_base if v in metro_df.columns]

# Promedios por segmento
segmento_perfil = metro_df.groupby('PRIMERA')[vars_cluster].mean()
# Agregar conteo
segmento_perfil['n_establecimientos'] = metro_df.groupby('PRIMERA').size()

print(f"Perfiles de segmentos: {segmento_perfil.shape}")
segmento_perfil

In [ ]:
# =============================================================================
# 4.2 Determinar número óptimo de clusters
# =============================================================================

# Escalar datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(segmento_perfil[vars_cluster].fillna(0))

# Método del codo + silhouette
K_range = range(2, min(8, len(segmento_perfil)))
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    if k > 1:
        silhouettes.append(silhouette_score(X_scaled, labels))
    else:
        silhouettes.append(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(list(K_range), inertias, 'bo-')
ax1.set_xlabel('Número de clusters (K)')
ax1.set_ylabel('Inercia')
ax1.set_title('Método del Codo')

ax2.plot(list(K_range), silhouettes, 'ro-')
ax2.set_xlabel('Número de clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score')

plt.tight_layout()
plt.savefig('fig_05_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

# Seleccionar K óptimo
k_optimo = list(K_range)[np.argmax(silhouettes)]
print(f"\n✅ K óptimo (máximo silhouette): {k_optimo}")

In [ ]:
# =============================================================================
# 4.3 Aplicar K-Means con K óptimo
# =============================================================================

km_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
segmento_perfil['cluster'] = km_final.fit_predict(X_scaled)

print("=" * 70)
print(f"CLUSTERING DE SEGMENTOS DEL METRO (K={k_optimo})")
print("=" * 70)

# Perfil de cada cluster
cluster_perfil = segmento_perfil.groupby('cluster')[vars_cluster + ['n_establecimientos']].mean()
print("\nPerfil promedio por cluster:")
print(cluster_perfil.round(3).to_string())

# Asignar nombres descriptivos basados en el perfil
print("\n\nSegmentos por cluster:")
for c in sorted(segmento_perfil['cluster'].unique()):
    segs = segmento_perfil[segmento_perfil['cluster'] == c].index.tolist()
    print(f"  Cluster {c}: Segmentos {segs}")

In [ ]:
# =============================================================================
# 4.4 Visualización del clustering con PCA
# =============================================================================

# PCA para visualización 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for c in sorted(segmento_perfil['cluster'].unique()):
    mask = segmento_perfil['cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
              c=colors[c], s=200, label=f'Cluster {c}', edgecolors='black', alpha=0.8)
    
    # Anotar cada punto con el número de segmento
    for idx, (x, y) in zip(segmento_perfil.index[mask], X_pca[mask]):
        ax.annotate(str(idx), (x, y), textcoords="offset points", 
                   xytext=(8, 8), fontsize=10, fontweight='bold')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.set_title('Clustering de Segmentos del Metro PLM por Perfil de Seguridad')
ax.legend(title='Cluster', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_06_clustering_metro.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Figura guardada: fig_06_clustering_metro.png")

In [ ]:
# =============================================================================
# 4.5 Heatmap de perfiles por cluster
# =============================================================================

# Normalizar para heatmap
cluster_norm = cluster_perfil[vars_cluster].copy()
for col in cluster_norm.columns:
    rng = cluster_norm[col].max() - cluster_norm[col].min()
    if rng > 0:
        cluster_norm[col] = (cluster_norm[col] - cluster_norm[col].min()) / rng

# Renombrar para legibilidad
rename_cols = {
    'P33G': 'Satisf. Seguridad',
    'seguridad_empeoro': '% Seg. Empeoró',
    'fue_victima': '% Víctima',
    'indice_satisfaccion': 'Satisf. Entorno',
    'P43_1A': 'Señalización',
    'P43_1B': 'Iluminación',
    'P43_1C': 'Residuos',
    'P43_1D': 'Reubic. SITP/TM',
    'P43_1E': 'Esp. Peatonales',
    'P43_1F': 'Hab. Calle',
    'P43_1G': 'Vigilancia Obra',
}
cluster_norm = cluster_norm.rename(columns=rename_cols)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cluster_norm.T, annot=True, cmap='RdYlGn', fmt='.2f', ax=ax,
            xticklabels=[f'Cluster {i}' for i in cluster_norm.index])
ax.set_title('Perfil de Seguridad por Cluster (valores normalizados 0-1)')
plt.tight_layout()
plt.savefig('fig_07_heatmap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Figura guardada: fig_07_heatmap_clusters.png")

---
## PARTE IV: ANÁLISIS COMPLEMENTARIO — CRUCE CON EMU Y EPV

In [ ]:
# =============================================================================
# 5.1 EMU: Percepción de seguridad en entorno urbano por localidad
# =============================================================================

# Seguridad en estaciones TransMilenio (P131_8) y entorno (P131_1 a P131_22)
vars_seg_entorno = [v for v in emu.columns if v.startswith('P131_')]

if 'Localidad' in emu.columns and len(vars_seg_entorno) > 0:
    seg_localidad = emu.groupby('Localidad')[vars_seg_entorno].mean()
    
    # Top/Bottom localidades en seguridad general (promedio de todos los P131)
    seg_localidad['seg_promedio'] = seg_localidad.mean(axis=1)
    
    print("--- LOCALIDADES CON MEJOR PERCEPCIÓN DE SEGURIDAD ---")
    print(seg_localidad['seg_promedio'].nlargest(5))
    
    print("\n--- LOCALIDADES CON PEOR PERCEPCIÓN DE SEGURIDAD ---")
    print(seg_localidad['seg_promedio'].nsmallest(5))

In [ ]:
# =============================================================================
# 5.2 Expectativas del Metro vs. Seguridad (EMU)
# =============================================================================

# ¿Las personas que esperan que el metro mejore la seguridad (P133_5)
# tienen diferente percepción actual de seguridad?

if 'P133_5' in emu.columns and 'P129' in emu.columns:
    emu_analisis = emu[['P133_5', 'P129']].dropna()
    
    # P133_5: Movilidad más segura (1=Sí, 2=No, 3=NsNr)
    grupo_si = emu_analisis.loc[emu_analisis['P133_5'] == 1, 'P129']
    grupo_no = emu_analisis.loc[emu_analisis['P133_5'] == 2, 'P129']
    
    if len(grupo_si) > 10 and len(grupo_no) > 10:
        t_stat, p_val = stats.ttest_ind(grupo_si, grupo_no)
        print(f"Satisfacción con Bogotá (P129):")
        print(f"  Creen que metro mejorará seguridad: {grupo_si.mean():.3f} (n={len(grupo_si)})")
        print(f"  No creen que mejorará seguridad:    {grupo_no.mean():.3f} (n={len(grupo_no)})")
        print(f"  t-stat: {t_stat:.3f}, p-valor: {p_val:.4f}")

In [ ]:
# =============================================================================
# 5.3 RNMC: Comparendos por UPZ cercanas a corredores
# =============================================================================

# Las UPZs del corredor de la Calle 13 y del Metro PLM
# (identificar manualmente las UPZs relevantes según la geografía)

# UPZs aproximadas del corredor del Metro (línea norte-sur por Av. Caracas/NQS)
# Esto depende de la geografía real; aquí se muestran como ejemplo
upzs_metro_aprox = [
    'AMERICAS', 'CIUDAD MONTES', 'RESTREPO', 'SOSIEGO',
    'LAS CRUCES', 'LA CANDELARIA', 'SANTA ISABEL',
    'LAS NIEVES', 'LA MACARENA', 'TEUSAQUILLO', 'GALERIAS',
    'CHAPINERO', 'CHICO LAGO'
]

upzs_calle13_aprox = [
    'ZONA INDUSTRIAL', 'PUENTE ARANDA', 'AMERICAS OCCIDENTAL',
    'FONTIBON', 'GRANJAS DE TECHO'
]

# NOTA: Ajustar estos nombres según los nombres reales en la base RNMC
print("UPZs únicas en RNMC (muestra):")
print(sorted(rnmc_upz['NOMBRE_UPZ'].unique())[:30])

print("\n⚠️ NOTA: Verificar manualmente qué UPZs corresponden a cada corredor")
print("para un análisis geográfico preciso.")

---
## PARTE V: RESUMEN DE HALLAZGOS Y RECOMENDACIONES

In [ ]:
# =============================================================================
# 6.1 Tabla resumen de hallazgos
# =============================================================================

print("=" * 70)
print("RESUMEN DE HALLAZGOS PRINCIPALES")
print("=" * 70)

hallazgos = [
    "1. DIFERENCIA ENTRE CORREDORES:",
    f"   - Metro PLM: {corredores.loc[corredores['TRAMO']==2, 'seguridad_empeoro'].mean()*100:.1f}% percibe empeoramiento",
    f"   - Calle 13:  {corredores.loc[corredores['TRAMO']==1, 'seguridad_empeoro'].mean()*100:.1f}% percibe empeoramiento",
    "",
    "2. VICTIMIZACIÓN:",
    f"   - Metro PLM: {corredores.loc[corredores['TRAMO']==2, 'fue_victima'].mean()*100:.1f}% fue víctima",
    f"   - Calle 13:  {corredores.loc[corredores['TRAMO']==1, 'fue_victima'].mean()*100:.1f}% fue víctima",
    "",
    "3. MiPymes:",
    f"   - % MiPymes en la muestra: {corredores['es_mipyme'].mean()*100:.1f}%",
    "",
    "4. CLUSTERS IDENTIFICADOS:",
    f"   - Se identificaron {k_optimo} grupos de tramos con perfiles diferenciados",
    "   - Permite diseñar intervenciones focalizadas por segmento",
]

for h in hallazgos:
    print(h)

In [ ]:
# =============================================================================
# 6.2 Marco de recomendaciones por cluster
# =============================================================================

print("=" * 70)
print("MARCO DE RECOMENDACIONES POR CLUSTER")
print("=" * 70)

for c in sorted(segmento_perfil['cluster'].unique()):
    segs = segmento_perfil[segmento_perfil['cluster'] == c].index.tolist()
    perfil = cluster_perfil.loc[c]
    
    print(f"\n--- CLUSTER {c} (Segmentos: {segs}) ---")
    
    if 'seguridad_empeoro' in perfil.index:
        if perfil['seguridad_empeoro'] > 0.5:
            print("  🔴 ALTA PERCEPCIÓN DE EMPEORAMIENTO")
            print("     → Priorizar: iluminación, vigilancia, presencia policial")
        elif perfil['seguridad_empeoro'] > 0.3:
            print("  🟠 PERCEPCIÓN MODERADA DE EMPEORAMIENTO")
            print("     → Priorizar: comunicación, señalización, acompañamiento a negocios")
        else:
            print("  🟢 BAJA PERCEPCIÓN DE EMPEORAMIENTO")
            print("     → Mantener: buenas prácticas de gestión de obra")
    
    if 'fue_victima' in perfil.index:
        if perfil['fue_victima'] > 0.3:
            print("  ⚠️ ALTA VICTIMIZACIÓN — Intervención urgente de seguridad")

print("\n" + "=" * 70)
print("FIN DEL ANÁLISIS — Usar estos resultados para el informe final")
print("=" * 70)